# Dataset : CIFAR-10  
            60,000 colour images (32×32×3) across 10 object classes:
            airplane, automobile, bird, cat, deer,
            dog, frog, horse, ship, truck
  Libraries: NumPy · Pandas · Matplotlib · Seaborn · Scikit-Learn · TensorFlow

In [ ]:
# Imports
import os, time, warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

import seaborn as sns
warnings.filterwarnings("ignore")


In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.decomposition   import PCA
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline        import Pipeline

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Output directory
OUT = "cifar10_outputs"
os.makedirs(OUT, exist_ok=True)

CLASS_NAMES = ["airplane","automobile","bird","cat","deer",
               "dog","frog","horse","ship","truck"]

In [ ]:
# Colour palette
BG_DARK  = "#0d0d1a"
BG_PANEL = "#16213e"
BG_MID   = "#1a1a2e"
ACCENT   = "#4C72B0"
GREEN    = "#2ecc71"
RED      = "#e74c3c"

def _style(fig, *axes):
    fig.patch.set_facecolor(BG_DARK)
    for ax in axes:
        ax.set_facecolor(BG_PANEL)
        ax.tick_params(colors="white")
        for sp in ax.spines.values():
            sp.set_edgecolor("#444466")

print("="*72)
print("  ML JOURNEY — CIFAR-10 Image Classification")
print(f"  TensorFlow {tf.__version__}  |  NumPy {np.__version__}  |  Pandas {pd.__version__}")
print("="*72)

results = {}   # name → {accuracy, train_time, history, type}




In [ ]:
#  DATA LOADING & EXPLORATORY DATA ANALYSIS
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.cifar10.load_data()
y_tr_raw = y_tr_raw.flatten()
y_te_raw = y_te_raw.flatten()

print(f"  Train : {X_tr_raw.shape}  |  Test : {X_te_raw.shape}")
print(f"  Pixel range : [{X_tr_raw.min()}, {X_tr_raw.max()}]")

In [ ]:
# Pandas EDA
flat = X_tr_raw.reshape(50_000, -1)          # 50000 × 3072
df_eda = pd.DataFrame({
    "label"   : y_tr_raw,
    "class"   : [CLASS_NAMES[i] for i in y_tr_raw],
    "mean_R"  : flat[:, 0:1024].mean(1),
    "mean_G"  : flat[:, 1024:2048].mean(1),
    "mean_B"  : flat[:, 2048:].mean(1),
    "brightness": flat.mean(1),
})
print("\n  Pandas head (first 5 rows):")
print(df_eda.head())
print("\n  Class distribution (training):")
print(df_eda["class"].value_counts().to_string())
print("\n  Channel statistics:")
print(df_eda[["mean_R","mean_G","mean_B","brightness"]].describe().round(2))


In [ ]:
# sample grid + class distribution
fig = plt.figure(figsize=(22, 11))
_style(fig)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Sample grid (3 × 10)
ax_grid = fig.add_subplot(gs[0, :2])
ax_grid.axis("off")
ax_grid.set_facecolor(BG_DARK)
ax_grid.set_title("Sample Images — 3 per Class", color="white", fontsize=14, pad=8)
for ci, cname in enumerate(CLASS_NAMES):
    idxs = np.where(y_tr_raw == ci)[0][:3]
    for j, idx in enumerate(idxs):
        ax_s = fig.add_axes([0.03 + ci*0.087, 0.58 - j*0.145, 0.075, 0.13])
        ax_s.imshow(X_tr_raw[idx])
        if j == 0:
            ax_s.set_title(cname, color="white", fontsize=7)
        ax_s.axis("off")
        
# Class distribution
ax_dist = fig.add_subplot(gs[0, 2])
_style(fig, ax_dist)
counts = df_eda["class"].value_counts().loc[CLASS_NAMES]
clrs   = sns.color_palette("plasma", 10)
ax_dist.barh(CLASS_NAMES, counts.values, color=clrs, edgecolor="white", lw=0.4)
ax_dist.set_title("Class Distribution", color="white", fontsize=13)
ax_dist.set_xlabel("Count", color="#aaaaaa")
ax_dist.tick_params(colors="white")
for v, bar in zip(counts.values, ax_dist.patches):
    ax_dist.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
                 f"{v:,}", va="center", color="white", fontsize=8)


In [ ]:
# Channel histograms
ax_ch = fig.add_subplot(gs[1, :2])
_style(fig, ax_ch)
sample = df_eda.sample(8000, random_state=SEED)
ax_ch.hist(sample["mean_R"], bins=50, color="red",   alpha=0.6, label="Red",   density=True)
ax_ch.hist(sample["mean_G"], bins=50, color="green", alpha=0.6, label="Green", density=True)
ax_ch.hist(sample["mean_B"], bins=50, color="#4488ff",alpha=0.6,label="Blue",  density=True)
ax_ch.set_title("Per-Channel Mean Intensity (8k samples)", color="white", fontsize=13)
ax_ch.set_xlabel("Mean Pixel Value", color="#aaaaaa")
ax_ch.set_ylabel("Density", color="#aaaaaa")
ax_ch.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")


In [ ]:
# Brightness violin
ax_vio = fig.add_subplot(gs[1, 2])
_style(fig, ax_vio)
sns.violinplot(data=df_eda.sample(5000, random_state=SEED),
               x="class", y="brightness", order=CLASS_NAMES,
               palette="plasma", inner="quartile", ax=ax_vio)
ax_vio.set_title("Brightness Distribution by Class", color="white", fontsize=12)
ax_vio.set_xlabel("", color="#aaaaaa")
ax_vio.set_ylabel("Mean Brightness", color="#aaaaaa")
ax_vio.set_xticklabels(CLASS_NAMES, rotation=40, ha="right",
                        color="white", fontsize=8)
ax_vio.tick_params(colors="white")

fig.suptitle("CIFAR-10 — Exploratory Data Analysis",
             color="white", fontsize=18, fontweight="bold", y=0.99)
plt.savefig(f"{OUT}/01_eda.png", dpi=150, bbox_inches="tight",
            facecolor=BG_DARK)
plt.close()
print(f"\n  [✓] EDA plot → {OUT}/01_eda.png")


In [ ]:
# PCA + mean class images

print("  Computing PCA …")
pca_viz  = PCA(n_components=50, random_state=SEED)
flat_n   = flat[:10_000] / 255.0
X_pca_v  = pca_viz.fit_transform(flat_n)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
_style(fig, *axes)

In [ ]:
# Scree plot
expl = pca_viz.explained_variance_ratio_ * 100
axes[0].bar(range(1, 51), expl, color=ACCENT, alpha=0.8)
axes[0].plot(range(1, 51), np.cumsum(expl), "w-o",
             lw=2, ms=4, label="Cumulative")
axes[0].axhline(80, color=GREEN, ls="--", lw=1.5, label="80%")
axes[0].set_title("PCA Explained Variance", color="white", fontsize=13)
axes[0].set_xlabel("PC", color="#aaaaaa")
axes[0].set_ylabel("Variance %", color="#aaaaaa")
axes[0].legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")


In [ ]:
# PC1 vs PC2 scatter
labels_sub = y_tr_raw[:10_000]
for ci in range(10):
    m = labels_sub == ci
    axes[1].scatter(X_pca_v[m, 0], X_pca_v[m, 1],
                    s=5, alpha=0.4, label=CLASS_NAMES[ci])
axes[1].set_title("PCA — PC1 vs PC2", color="white", fontsize=13)
axes[1].set_xlabel("PC1", color="#aaaaaa")
axes[1].set_ylabel("PC2", color="#aaaaaa")
legend = axes[1].legend(markerscale=3, fontsize=7,
                         facecolor="#2a2a4a", edgecolor="white",
                         labelcolor="white", ncol=2)


In [ ]:
# Mean images
axes[2].axis("off")
axes[2].set_title("Mean Image per Class", color="white", fontsize=13)
for ci in range(10):
    mean_img = X_tr_raw[y_tr_raw == ci].mean(0).astype(np.uint8)
    ax_m = fig.add_axes([0.68 + (ci % 5)*0.058,
                          0.08 + (1 - ci//5)*0.38,
                          0.05, 0.3])
    ax_m.imshow(mean_img)
    ax_m.set_title(CLASS_NAMES[ci], color="white", fontsize=7)
    ax_m.axis("off")

fig.suptitle("CIFAR-10 — PCA & Mean Images",
             color="white", fontsize=16, fontweight="bold")
plt.savefig(f"{OUT}/02_pca.png", dpi=150, bbox_inches="tight",
            facecolor=BG_DARK)
plt.close()
print(f"  [✓] PCA plot → {OUT}/02_pca.png")

In [ ]:
# DATA PREPROCESSING
# Normalized float arrays 
X_tr_f  = X_tr_raw.astype("float32") / 255.0   # (50000, 32, 32, 3)
X_te_f  = X_te_raw.astype("float32") / 255.0   # (10000, 32, 32, 3)



In [ ]:
# Flattened (for sklearn)
X_tr_flat = X_tr_f.reshape(50_000, -1)          # (50000, 3072)
X_te_flat = X_te_f.reshape(10_000, -1)          # (10000, 3072)

In [ ]:
# One-hot labels 
y_tr_oh  = keras.utils.to_categorical(y_tr_raw, N_CLASSES)
y_te_oh  = keras.utils.to_categorical(y_te_raw, N_CLASSES)

In [ ]:
# Sklearn subset
SK_TR = 15_000;  SK_TE = 3_000
X_sk_tr = X_tr_flat[:SK_TR];  y_sk_tr = y_tr_raw[:SK_TR]
X_sk_te = X_te_flat[:SK_TE];  y_sk_te = y_te_raw[:SK_TE]